# 04.01 - Bring-your-own LLM: preparing a model for LLiMa

**Why.** The `llima` catalog already covers the common families, so most of the time you just
`llima pull` a ready model (see `01-llima-basics` and `02-run-llm-vlm`). This notebook is for the
other case: you have a model that is **not** in the catalog - a private fine-tune, or a family/size
combination SiMa has not published - and you need to get it onto the Modalix DevKit.

This is a **concept + documented-procedure** notebook, not a runnable one. Every heavy command is a
printed Python string so that running a cell can never launch a compile or a download.

## Honesty first: there is no `llima compile`

The on-board `llima` CLI has exactly these subcommands:

```text
llima {run, search, pull, list, rm, benchmark-server}
```

**None of them compiles a model.** `llima` is a *runtime + model-manager* CLI: it searches the
remote catalog, pulls already-compiled models, lists/removes local ones, and quick-tests them
(`run`, `benchmark-server`). If you type `llima compile ...` it will fail - that command does not
exist.

Bring-your-own compilation is a **separate, host-side Model Compiler step**, documented by SiMa as
the **`llima-compile`** tool:

- https://developer.sima.ai/software/genai-llima/compilation_genai (compile flow)
- https://developer.sima.ai/software/genai-llima/ (supported families, input formats)
- [`compatibility.mdx`](https://github.com/sima-neat/core/blob/main/docs/getting-started/compatibility.mdx): *"The Model Compiler is
  required when you compile or quantize models yourself, including GenAI models."*

> **A note on certainty.** `llima-compile` is a host-side Model Compiler tool, separate from the
> on-board `llima` CLI. The runtime-contract facts below come from Neat core source and are firm;
> the exact `llima-compile` flags come from the official SiMa docs and should be confirmed against
> the current documentation before you rely on them. A wrong-but-plausible command taught to a
> class is worse than an honest boundary.

## Where compilation sits in the flow

```text
  (host / workstation, GPU recommended)         (Modalix DevKit)
  your model  --->  llima-compile  --->  model dir  --->  llima pull / copy  --->  llima run
   (HF / GGUF /                        (devkit/ +           to board            or pyneat.genai
    GPTQ-AWQ)                           elf_files/)                              GenAIModel / server
```

The heavy lifting (ONNX export, quantization, MLA compile) happens **off** the board. The board only
ever sees the finished model directory. That is why `llima` itself has no compile verb.

## Supported LLM architectures

`llima-compile` targets a fixed set of transformer families. From the LLiMa docs plus the live
catalog (`llima search`):

| Family | Examples in catalog |
| --- | --- |
| Llama 2 / 3.1 / 3.2 | `Llama-3.1-8B-Instruct-a16w4`, `Llama-3.2-3B-Instruct-a16w4` |
| Gemma 1 / 2 / 3 / 4 | `gemma-3-4b-it-a16w4`, `gemma-4-E4B-it-GPTQ-a16w4` |
| Phi-3.5 | `Phi-3.5-mini-instruct-a16w4` |
| Qwen2.5 / Qwen3 | `Qwen3-4B-Instruct-2507-GPTQ-a16w4` |
| Mistral | `Mistral-7B-Instruct-v0.3-a16w4` |
| LFM2 / LFM2.5 | `LFM2-1.2B-a16w4`, `LFM2.5-1.2B-Instruct-a16w4` |

**Rule of thumb:** it is the *architecture family*, not the checkpoint name, that has to be
supported. A LoRA/SFT fine-tune of a supported base (same layer structure) is normally fine; a novel
architecture is not. If your model is not a member of one of these families, stop and confirm against
the official docs - do not assume it compiles. (VLM families are covered in `02_vlm_compilation.ipynb`.)

## Accepted source formats

`llima-compile` accepts three input shapes, each with its own internal pipeline:

| Source format | Internal pipeline | Notes |
| --- | --- | --- |
| **Hugging Face safetensors** (full precision) | export to ONNX -> quantize -> compile | needs the companion config files (below) |
| **GGUF** | direct -> compile | already a self-contained single file |
| **Pre-quantized GPTQ / AWQ** (compressed-tensors) | source-to-quant -> compile | a symmetry requirement may apply; confirm against the docs |

A Hugging Face source directory must contain:

```text
config.json
tokenizer.json
tokenizer_config.json
generation_config.json
<weights>.safetensors
```

If any of these is missing, fix the inputs before attempting a compile.

## Quantization: what `a16w4` / `a16w8` mean, and the compile knob

The model-ID suffix encodes the on-device quantization:

- **`a16w4`** = 16-bit activations, **4-bit** weights (smallest, fastest, lowest memory)
- **`a16w8`** = 16-bit activations, **8-bit** weights (larger, higher fidelity)

At compile time the docs expose a `precision` setting with these values:

| Precision value | Corresponds to |
| --- | --- |
| `BF16` | full precision, no weight quantization |
| `A_BF16_W_INT8` | ~ the `a16w8` suffix |
| `A_BF16_W_INT4` | ~ the `a16w4` suffix |

Pick the **lowest** precision that still meets your accuracy bar - `a16w4` roughly halves the weight
footprint versus `a16w8`, which matters a lot on a board with limited free space.

### The compile command (documented, NOT executed)

Below is the documented shape only. It is a printed string; running this cell prints text and
launches nothing. This step runs on a host with the Model Compiler installed (GPU recommended),
not on the board.

In [ ]:
# Documented shape only, from
# https://developer.sima.ai/software/genai-llima/compilation_genai
# The exact flags belong to the Model Compiler package (llima-compile), a host-side
# tool. Confirm them against the current official SiMa docs before relying on them.

compile_cmd = (
    "llima-compile [options] <path-to-hf-or-gguf-or-gptq-model>"
)

# Documented behaviour to expect (confirm against the official docs):
#   * HF safetensors  -> exports ONNX, quantizes, then compiles
#   * GGUF            -> compiles directly
#   * GPTQ / AWQ      -> source-to-quant, then compiles (may require symmetric quant; confirm in docs)
#   * precision knob  -> one of {BF16, A_BF16_W_INT8, A_BF16_W_INT4}
#   * context length  -> --max_num_tokens (default 1024)

print("Documented command shape (runs on a host with the Model Compiler, not the board):")
print(" ", compile_cmd)

## Compile flow, end to end (documented procedure)

1. **Pick / prepare the source** on a host workstation (Step 0-3 of `notes/triage_checklist.md`):
   confirm the architecture family is supported, the format is accepted, and (HF path) the four
   companion config files + safetensors are present.
2. **Run `llima-compile`** with your chosen `precision`. HF inputs go through
   ONNX -> quantize -> compile; GGUF compiles directly; GPTQ/AWQ go through source-to-quant. The
   quantize step is the heavy one - a GPU is recommended.
3. **Inspect the output tree** (next section) and confirm it produced the runtime-required pieces.
4. **Deploy to the board**: place the finished model directory under
   `/media/nvme/llima/models/<model-id>` (the same location `llima pull` writes to).
5. **Smoke-test on the board** with the cheap runtime path:
   `llima run <model-id>` or `pyneat.genai.GenAIModel(<dir>)`. See `02-run-llm-vlm`.

## Resulting artifact layout on disk

Two layouts matter, and they are **not the same**. Keep them straight.

**A. Compiler output tree** - what `llima-compile` writes on the host:

```text
output_directory/
  onnx_files/            # ONNX intermediates (HF path only)
  sima_files/
    devkit/              # Python runtime / config files
    mpk/                 # compiled MLA binaries
    npy_files/           # LoRA adapter weights (only if compiled with LoRA)
```

**B. Deployed model directory** - what the Neat GenAI runtime actually *loads* from, verified
in [`GenAIInternal.cpp`](https://github.com/sima-neat/core/blob/main/src/genai/GenAIInternal.cpp) (`inspect_model_directory`):

```text
/media/nvme/llima/models/<model-id>/
  devkit/
    vlm_config.json      # present for LLM and VLM  (exactly one of these two)
    whisper_config.json  # present for ASR
  elf_files/             # the compiled MLA binaries  (mpk/ contents, deployed)
```

The runtime **throws** if `devkit/` is missing, if `elf_files/` is missing, or if `devkit/` has both
config files or neither. Deployment reorganizes the compiler's `sima_files/` into this
`devkit/` + `elf_files/` shape. That contract is the single most useful thing to verify after any
compile - see Step 5 of the triage checklist.

## Constraints and gotchas

- **`llima` cannot compile.** Compilation is `llima-compile` (Model Compiler), host-side.
- **Board disk is tiny.** The DevKit root filesystem is small; check it with `df -h /`. Deployed
  weights go on `/media/nvme` (larger), but always check both before deploying. Prefer `a16w4` and
  the smallest viable variant.
- **Default context is 1024 tokens** (`--max_num_tokens`), configurable at compile time. Longer context costs
  runtime KV-cache memory.
- **GPTQ/AWQ may need to be symmetric** - the compile docs list GPTQ/AWQ as an input format but do not spell out the symmetry rule; confirm against the current docs.
- **Verify the output contract** (`devkit/` + `elf_files/`, exactly one config) or the model will not
  load - this is enforced in the Neat runtime, not just documented.

In [ ]:
# Safe precheck to run BEFORE any deploy/pull on the board (cheap, no model run).
# This only inspects disk; shown as a printed string.
disk_precheck = "df -h / && df -h /media/nvme"
print("Run on the board before deploying weights:")
print(" ", disk_precheck)

## Expected artifacts

After a successful `llima-compile` + deploy, you should be able to confirm:

- **Compiler host output** contains `sima_files/devkit/` and `sima_files/mpk/` (plus `onnx_files/`
  for the HF path).
- **Deployed** `/media/nvme/llima/models/<model-id>/` contains:
  - `devkit/vlm_config.json` (LLM/VLM) **xor** `devkit/whisper_config.json` (ASR)
  - `elf_files/` with the compiled binaries
- `llima list` on the board shows `<model-id>`.
- `pyneat.genai.GenAIModel("/media/nvme/llima/models/<model-id>")` constructs without throwing, and
  `model.accepts_text()` is `True`. (See `02-run-llm-vlm/01_run_llm.ipynb`.)

If construction throws `"missing devkit/"` or `"missing elf_files/"`, the compile/deploy did not
produce the runtime contract - go back to the artifact-layout section.